# 10 Nonlinear controller

This section investigates two nonlinear control strategies that impose physically meaningful constraints on the HVAC heat rate $q_{HVAC}$, making them better suited to real building systems. The first strategy, the Proportional Nonlinear Controller (PNC), computes the HVAC command proportionally to the instantaneous temperature error. The second, the Proportional-Integral with saturation Nonlinear Controller (PINC), includes an integral term, an anti-windup mechanism, and a saturation law, allowing the controller to eliminate steady-state offset while preventing unbounded accumulation of the integral state and keeping the controller output within physical limits. 

Both controllers are evaluated across five representative scenarios: winter heating, summer cooling, combined heating and cooling, solar protection, and passive cooling. The results are analyzed in terms of thermal comfort, energy consumption, and HVAC power smoothness. 

These graphs establish the reference: without any HVAC, the indoor temperature reaches 33.5 °C in summer and drops to -2.9 °C in winter, both far outside the comfort band. The heat rate panels confirm that $q_{HVAC} = 0$ throughout, meaning all thermal dynamics are driven purely by the building envelope and solar gains. 

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import dm4bem


def control_for(controller, period, dt=30, nonlinear_controller=True):
    # =========================================================
    # Obtain state-space representation
    # =========================================================

    # Disassembled thermal circuits
    folder_path = 'bldg2'
    TCd = dm4bem.bldg2TCd(folder_path, TC_auto_number=True)

    # Set HVAC controller gain (G=0 in TC3.csv placeholder)
    #TCd['c3']['G']['c3_q0'] = 1e3       # Kp [W/K]
    # TCd['c2']['C']['c2_θ0'] = 0       # zero indoor air capacity
    # TCd['c0']['C']['c0_θ0'] = 0       # zero glass capacity

    # Assembled thermal circuit
    ass_lists  = pd.read_csv('bldg2/assembly_lists.csv')
    ass_matrix = dm4bem.assemble_lists2matrix(ass_lists)
    TC         = dm4bem.assemble_TCd_matrix(TCd, ass_matrix)

    # State-space
    [As, Bs, Cs, Ds, us] = dm4bem.tc2ss(TC)

    # =========================================================
    # Eigenvalue analysis — maximum stable time step
    # =========================================================
    λ      = np.linalg.eig(As)[0]
    dt_max = 2 * min(-1. / λ)
    dt_max = dm4bem.round_time(dt_max)

    # =========================================================
    # Simulation with weather data
    # =========================================================
    start_date = '2025-' + period[0]
    end_date   = '2025-' + period[1]

    # Weather
    filename = './weather_data/FRA_AR_Grenoble.Alpes.Isere.AP.074860_TMYx.2011-2025.epw'
    [data, meta] = dm4bem.read_epw(filename, coerce_year=None)
    weather = data[["temp_air", "dir_n_rad", "dif_h_rad"]]
    del data
    weather.index = weather.index.map(lambda t: t.replace(year=2025))
    weather = weather.loc[start_date:end_date]

    # Temperature sources
    To = weather['temp_air']

    Ti_day, Ti_night = 20, 16
    Ti_sp = pd.Series(
        [Ti_day if 6 <= hour <= 22 else Ti_night
         for hour in To.index.hour],
        index=To.index)

    # Flow-rate sources — solar irradiance on south vertical wall
    wall_out = pd.read_csv('bldg2/walls_out.csv')
    w0 = wall_out[wall_out['ID'] == 'w0']

    surface_orientation = {'slope':   w0['β'].values[0],
                           'azimuth': w0['γ'].values[0],
                           'latitude': 45}

    rad_surf = dm4bem.sol_rad_tilt_surf(
        weather, surface_orientation, w0['albedo'].values[0])

    Etot = rad_surf.sum(axis=1)

    # Window / door optical properties
    α_gSW = 0.38    # short-wave absorptivity: reflective blue glass
    τ_gSW = 0.30    # short-wave transmittance: reflective blue glass
    S_g   = 3.6     # m2, window glass area
    S_b   = 2.7     # m2, door area

    # Flow-rate sources
    Φo = w0['α0'].values[0] * w0['Area'].values[0] * Etot   # solar on outdoor wall
    Φi = τ_gSW * w0['α0'].values[0] * S_g * Etot            # solar through glass → indoor air
    Φa = α_gSW * S_g * Etot                                  # solar absorbed by glass
    Φd = w0['α0'].values[0] * S_b * Etot                     # solar on door

    # Auxiliary (internal) heat sources
    Qa = pd.Series(0, index=To.index)

    # Input data set
    input_data_set = pd.DataFrame({'To':    To,
                                   'Ti_sp': Ti_sp,
                                   'Φo':    Φo,
                                   'Φi':    Φi,
                                   'Qa':    Qa,
                                   'Φa':    Φa,
                                   'Φd':    Φd,
                                   'Etot':  Etot})

    # =========================================================
    # Time integration
    # =========================================================
    # Resample hourly data to time step dt
    input_data_set = input_data_set.resample(
        str(dt) + 'S').interpolate(method='linear')

    # Get input vector in time from input_data_set
    u = dm4bem.inputs_in_time(us, input_data_set)

    # Initial conditions
    θ0    = 20.0
    θ_exp = pd.DataFrame(index=u.index)
    θ_exp[As.columns] = θ0

    # Euler explicit time integration
    I = np.eye(As.shape[0])
    for k in range(1, u.shape[0] - 1):
        if nonlinear_controller:
            exec(controller)
        θ_exp.iloc[k + 1] = (I + dt * As) @ θ_exp.iloc[k] \
                             + dt * Bs @ u.iloc[k]

    # Outputs
    y = (Cs @ θ_exp.T + Ds @ u.T).T

    Kp = TC['G']['c3_q0']   # W/K, controller gain
    S  = 30                  # m², room floor area

    # q_HVAC [W/m²]
    if nonlinear_controller:
        q_HVAC = u['c2_θ0']
    else:
        q_HVAC = Kp * (u['c3_q0'] - y['c2_θ0']) / S

    # Plot
    data = pd.DataFrame({'To':     input_data_set['To'],
                         'θi':     y['c2_θ0'],
                         'Etot':   input_data_set['Etot'],
                         'q_HVAC': q_HVAC})

    fig, axs = plt.subplots(2, 1, sharex=True)
    data[['To', 'θi']].plot(ax=axs[0],
                             xticks=[],
                             ylabel='Temperature, $θ$ / °C')
    axs[0].legend(['$θ_{outdoor}$', '$θ_{indoor}$'], loc='upper right')
    axs[0].grid(True)

    data[['Etot', 'q_HVAC']].plot(ax=axs[1],
                                   ylabel='Heat rate, $q$ / (W·m⁻²)')
    axs[1].set(xlabel='Time')
    axs[1].legend(['$E_{total}$', '$q_{HVAC}$'], loc='upper right')
    axs[1].grid(True)
    print(TC['C']['c2_θ0'])
    plt.show()


In [ ]:

#################
# Free running winter  #
#################
start_date = '01-01 12:00:00'
end_date   = '01-31 12:00:00'
period     = [start_date, end_date]

control_for("free running", period, dt=30, nonlinear_controller=False)


In [ ]:

#################
# Free running summer  #
#################
start_date = '07-01 12:00:00'
end_date   = '07-31 12:00:00'
period     = [start_date, end_date]

control_for("free running", period, dt=30, nonlinear_controller=False)


## 10.1 Proportional nonlinear controller (PNC)

### Heating

The PNC drives $\theta_{indoor}$ upward and holds it near the 20 °C setpoint, but the heat rate panel shows sustained high $q_{HVAC}$, peaking near 1500 W·m^-2. Because the proportional controller acts directly on the instantaneous error $e =\theta_{set} -\theta_{indoor}$, it applies strong corrective action when the outdoor-indoor temperature gap is large. 


In [ ]:
# =============================================================
# Heating
# =============================================================
heating = """
Tisp = 20       # indoor setpoint temperature, °C
Kpp  = 1e3      # controller gain

if Tisp < θ_exp.iloc[k - 1]['c2_θ0']:
    u.iloc[k]['c2_θ0'] = 0
else:
    u.iloc[k]['c2_θ0'] = Kpp * (Tisp - θ_exp.iloc[k - 1]['c2_θ0'])
"""
start_date = '01-01 12:00:00'
end_date   = '01-31 12:00:00'
period     = [start_date, end_date]
control_for(heating, period)


### Cooling

In summer, the heat rate panel shows negative $q_{HVAC}$ values during the hottest daytime periods, with magnitudes up to about -500 W·m^-2. The controller returns to zero during cooler nights, which shows that the PNC reacts only to immediate cooling demand. 


In [ ]:
# =============================================================
# Cooling
# =============================================================
cooling = """
Tisp = 20       # indoor setpoint temperature, °C
Δθ   = 5        # temperature deadband, °C
Kpp  = 1e2      # controller gain

if θ_exp.iloc[k - 1]['c2_θ0'] < Tisp + Δθ:
    u.iloc[k]['c2_θ0'] = 0
else:
    u.iloc[k]['c2_θ0'] = Kpp * (Tisp - θ_exp.iloc[k - 1]['c2_θ0'])
"""
period = ['07-01 12:00:00', '07-31 12:00:00']
control_for(cooling, period)




### Heating and cooling

In summer mixed conditions, the PNC alternates between cooling and heating as outdoor temperature oscillates. The heat rate swings from roughly -1500 to +500 W·m^-2, which indicates chattering behavior and unnecessary energy use caused by a memoryless proportional law. 


In [ ]:
# =============================================================
# Heating and cooling with deadband
# =============================================================
heat_cool = """
Tisp = 20       # indoor setpoint temperature, °C
Δθ   = 5        # temperature deadband, °C
Kpp  = 3e2      # controller gain

if Tisp < θ_exp.iloc[k - 1]['c2_θ0'] < Tisp + Δθ:
    u.iloc[k]['c2_θ0'] = 0
else:
    u.iloc[k]['c2_θ0'] = Kpp * (Tisp - θ_exp.iloc[k - 1]['c2_θ0'])
"""
period = ['07-01 12:00:00', '07-31 12:00:00']
control_for(heat_cool, period)



### Solar protection

The PNC responds to afternoon solar gain spikes by increasing the cooling output and keeping$\theta_{indoor}$ below 27.5 °C. The transitions between cooling and zero output remain relatively clean, but the setpoint is not maintained perfectly. 


In [ ]:


# =============================================================
# Solar protection
# =============================================================
solar_protection = """
Tisp = 20       # indoor setpoint temperature, °C
Δθ   = 1        # temperature deadband, °C
Kpp  = 3e2      # controller gain

if θ_exp.iloc[k - 1]['c2_θ0'] < Tisp + Δθ:
    u.iloc[k]['c2_θ0'] = 0
else:
    u.iloc[k]['c2_θ0'] = Kpp * (Tisp - θ_exp.iloc[k - 1]['c2_θ0'])
    u.iloc[k]['c2_θ0'] = min(u.iloc[k]['c2_θ0'], 0)
    u.iloc[k]['ow0_θ0'] *= 0.1
"""
period = ['07-15 12:00:00', '07-17 12:00:00']
control_for(solar_protection, period)


### Passive cooling

During passive cooling, the PNC correctly deactivates HVAC when natural ventilation is sufficient, with $q_{HVAC}$ staying near zero for long periods. The indoor temperature remains roughly within 20–27 °C, although the controller still makes occasional unnecessary corrections because of its reactive nature. 

In [ ]:
# =============================================================
# Passive cooling
# =============================================================
passive_cooling = """
Tisp  = 20.0    # indoor setpoint temperature, °C
Δθ    = 1.0     # temperature deadband, °C

l      = 30.0   # m², room floor area
H      = 3.0    # m, room height
Va     = l * H  # m³, room volume
ACH    = 10.0   # 1/h, air changes per hour (night ventilation)
Va_dot = ACH / 3600 * Va
ρ      = 1.2    # kg/m³
c      = 1000.0 # J/(kg·K)
G_free = ρ * c * Va_dot
q_free = G_free * (u.iloc[k - 1]['ow0_q0'] - θ_exp.iloc[k - 1]['c2_θ0'])

if θ_exp.iloc[k - 1]['c2_θ0'] < Tisp + Δθ:
    u.iloc[k]['c2_θ0'] = 0
else:
    u.iloc[k]['c2_θ0'] = 0
    if u.iloc[k - 1]['ow0_q0'] < Tisp:
        u.iloc[k]['c2_θ0']  = q_free
        u.iloc[k]['ow0_θ0'] *= 0.1
"""
period = ['07-15 12:00:00', '07-17 12:00:00']
control_for(passive_cooling, period)

## 10.2 Proportional-Integral with saturation nonlinear controller (PINC)


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import dm4bem

# =============================================================
# Physical parameters  (single source of truth)
# =============================================================
materials = {
    'concrete':   {'lam': 1.4,   'rho': 2300, 'c': 880,  'w': 0.32},
    'insulation': {'lam': 0.027, 'rho': 55,   'c': 1210, 'w': 0.08},
    'glass':      {'lam': 1.4,   'rho': 2500, 'c': 1210, 'w': 0.04},
    'wood':       {'lam': 0.2,   'rho': 310,  'c': 1733, 'w': 0.07},
    'air':        {              'rho': 1.2,   'c': 1000.0          },
}

# -- Surfaces [m2]
S_c = 26.7   # outdoor wall
S_g = 3.6    # window glass
S_b = 2.7    # door

# -- Room geometry
A_floor = 30.0   # m2
H       = 3.0    # m
V_a     = A_floor * H   # m3

# -- Convection coefficients [W/m2K]
h_o = 25.0
h_i = 8.0

# -- Ventilation
ACH   = 1.0
V_dot = ACH / 3600.0 * V_a   # m3/s

# -- Optical properties
alpha_wSW = 0.25
alpha_gSW = 0.38
tau_gSW   = 0.30

# -- Design solar irradiance for f-row labels [W/m2]
E_d = 200.0

# -- HVAC
Q_MAX   = 2000.0   # W, saturation limit
Kp_hvac = 1e3      # W/K, proportional gain

# =============================================================
# Derived conductances  [W/K]
# =============================================================
G0  = h_o * S_c
G1  = 2 * materials['concrete']['lam']   * S_c / materials['concrete']['w']
G2  = 2 * materials['concrete']['lam']   * S_c / materials['concrete']['w']
G3  = 2 * materials['insulation']['lam'] * S_c / materials['insulation']['w']
G4  = 2 * materials['insulation']['lam'] * S_c / materials['insulation']['w']
G5  = h_i * S_c
G6  = h_i * S_g
G7  = materials['glass']['lam'] * S_g / materials['glass']['w']
G8  = h_o * S_g
G9  = h_o * S_b
G10 = 2 * materials['wood']['lam'] * S_b / materials['wood']['w']
G11 = 2 * materials['wood']['lam'] * S_b / materials['wood']['w']
G12 = materials['air']['rho'] * materials['air']['c'] * V_dot

# =============================================================
# Derived capacitances  [J/K]
# =============================================================
C1 = materials['concrete']['rho']   * materials['concrete']['c']   * S_c * materials['concrete']['w']
C3 = materials['insulation']['rho'] * materials['insulation']['c'] * S_c * materials['insulation']['w']
C5 = materials['air']['rho']        * materials['air']['c']        * V_a
C7 = materials['glass']['rho']      * materials['glass']['c']      * S_g * materials['glass']['w']
C8 = materials['wood']['rho']       * materials['wood']['c']       * S_b * materials['wood']['w']

# =============================================================
# f-row label values
# =============================================================
Phi_o = alpha_wSW * S_c * E_d
Phi_i = tau_gSW * alpha_wSW * S_g * E_d
Phi_a = alpha_gSW * S_g * E_d
Phi_d = alpha_wSW * S_b * E_d


# =============================================================
# Main simulation function
# =============================================================
def control_for(controller, period, dt=30, nonlinear_controller=True):

    # Obtain state-space representation
    # ==================================

    # -- Disassembled thermal circuits (TCd)
    #    base.py uses:  TCd = dm4bem.bldg2TCd(folder_path, TC_auto_number=True)
    #    Here we have a single TCgen.csv, so we load it as one TC
    #    and wrap it into the same TCd dict structure.
    TCd = dm4bem.file2TC('TCgen.csv', name='c0', auto_number=False)

    # Override every G and C with the dict-derived values
    # (CSV holds numbers only as placeholders for dm4bem to parse)
    TCd['G']['c0q0']  = G0
    TCd['G']['c0q1']  = G1
    TCd['G']['c0q2']  = G2
    TCd['G']['c0q3']  = G3
    TCd['G']['c0q4']  = G4
    TCd['G']['c0q5']  = G5
    TCd['G']['c0q6']  = G6
    TCd['G']['c0q7']  = G7
    TCd['G']['c0q8']  = G8
    TCd['G']['c0q9']  = G9
    TCd['G']['c0q10'] = G10
    TCd['G']['c0q11'] = G11
    TCd['G']['c0q12'] = G12
    TCd['G']['c0q13'] = Kp_hvac   # HVAC controller gain

    TCd['C']['c0θ1']  = C1
    TCd['C']['c0θ3']  = C3
    TCd['C']['c0θ5']  = C5
    TCd['C']['c0θ7']  = C7
    TCd['C']['c0θ8']  = C8

    TCd['f']['c0θ5']  = 'Qaux'   # HVAC injection at indoor air node

    # -- Assembled thermal circuit
    #    base.py uses:  ass_lists  = pd.read_csv('bldg/assembly_lists.csv')
    #                   ass_matrix = dm4bem.assemble_lists2matrix(ass_lists)
    #                   TC = dm4bem.assemble_TCd_matrix(TCd, ass_matrix)
    #    With a single TC there are no inter-circuit connections,
    #    so the assembled TC is identical to TCd.
    TC = TCd

    # -- State-space representation
    [As, Bs, Cs, Ds, us] = dm4bem.tc2ss(TC)

    # -- Eigenvalue analysis
    λ      = np.linalg.eig(As)[0]
    dt_max = 2 * min(-1. / λ)
    dt_max = dm4bem.round_time(dt_max)

    # Simulation with weather data
    # ============================
    start_date = '2000-' + period[0]
    end_date   = '2000-' + period[1]

    filename = './weather_data/FRA_AR_Grenoble.Alpes.Isere.AP.074860_TMYx.2011-2025.epw'
    [data, meta] = dm4bem.read_epw(filename, coerce_year=None)
    weather = data[["temp_air", "dir_n_rad", "dif_h_rad"]]
    del data
    weather.index = weather.index.map(lambda t: t.replace(year=2000))
    weather = weather.loc[start_date:end_date]

    # Temperature sources
    To = weather['temp_air']

    Ti_day, Ti_night = 20, 16
    Ti_sp = pd.Series(
        [Ti_day if 6 <= hour <= 22 else Ti_night
         for hour in To.index.hour],
        index=To.index)

    # Flow-rate sources
    # base.py reads wall_out from walls_out.csv; here we use the
    # same surface orientation parameters directly
    surface_orientation = {'slope': 90, 'azimuth': 0, 'latitude': 45}
    rad_surf = dm4bem.sol_rad_tilt_surf(weather, surface_orientation,
                                        albedo=0.2)
    Etot = rad_surf.sum(axis=1)

    # Solar radiation flow sources (time-varying, scaled by Etot)
    Φo = alpha_wSW * S_c * Etot          # solar absorbed by outdoor wall
    Φi = tau_gSW * alpha_wSW * S_g * Etot   # solar transmitted through glass
    Φa = alpha_gSW * S_g * Etot          # solar absorbed by glass
    Φd = alpha_wSW * S_b * Etot          # solar absorbed by door

    # Auxiliary (internal) heat source
    Qa = pd.Series(0, index=To.index)

    # Input data set
    # Numeric keys match the f-row values in TCgen.csv (Phi_o, Phi_i, etc.)
    input_data_set = pd.DataFrame({
        'To':    To,
        'Ti':    Ti_sp,
        Phi_o:   Φo,
        Phi_i:   Φi,
        'Qaux':  Qa,
        Phi_a:   Φa,
        Phi_d:   Φd,
        'Etot':  Etot,
    })

    # Time integration
    # ----------------
    # Resample hourly data to time step dt
    input_data_set = input_data_set.resample(
        str(dt) + 'S').interpolate(method='linear')

    # Get input vector from input_data_set
    u = dm4bem.inputs_in_time(us, input_data_set)

    # Initial conditions
    θ0    = 20.0
    θ_exp = pd.DataFrame(index=u.index)
    θ_exp[As.columns] = θ0

    integral = [0.0]

    # Euler explicit time integration
    I = np.eye(As.shape[0])
    for k in range(1, u.shape[0] - 1):
        if nonlinear_controller:
            exec(controller)
        θ_exp.iloc[k + 1] = (I + dt * As) @ θ_exp.iloc[k] \
                             + dt * Bs @ u.iloc[k]

    # Outputs
    y = (Cs @ θ_exp.T + Ds @ u.T).T

    Kp = TC['G']['c0q13']   # W/K, controller gain

    if nonlinear_controller:
        q_HVAC = u['c0θ5']
    else:
        q_HVAC = Kp * (u['c0q13'] - y['c0θ5']) / A_floor

    # Plot
    data = pd.DataFrame({
        'To':     input_data_set['To'],
        'θi':     y['c0θ5'],
        'Etot':   input_data_set['Etot'],
        'q_HVAC': q_HVAC,
    })

    fig, axs = plt.subplots(2, 1, sharex=True)
    data[['To', 'θi']].plot(ax=axs[0], xticks=[],
                             ylabel='Temperature, $θ$ / °C')
    axs[0].legend(['$θ_{outdoor}$', '$θ_{indoor}$'], loc='upper right')
    axs[0].grid(True)
    data[['Etot', 'q_HVAC']].plot(ax=axs[1],
                                   ylabel='Heat rate, $q$ / (W·m⁻²)')
    axs[1].set(xlabel='Time')
    axs[1].legend(['$E_{total}$', '$q_{HVAC}$'], loc='upper right')
    axs[1].grid(True)
    print(TC['C']['c0θ5'])
    plt.show()




### Heating

The PINC shows a winter profile similar to the PNC, with indoor temperature stable near the setpoint and heat rate ranging from 0 to 1500 W·m^-2. The integral term accumulates the error over time and removes the steady-state offset, so$\theta_{indoor}$ locks onto the setpoint more accurately. 


In [ ]:

#################
# Heating       #
#################
heating = """
Tisp = 20       # indoor setpoint temperature, °C
Kpp  = 1e3      # controller gain
Ki   = 0.1      # integral gain
q_max = Q_MAX

error = Tisp - θ_exp.iloc[k - 1]['c0θ5']
u_pi  = Kpp * error + Ki * integral[0]
u_sat = float(np.clip(u_pi, 0, q_max))
u.iloc[k]['c0θ5'] = u_sat

if 0 < u_pi < q_max:
    integral[0] = integral[0] + error * dt
"""
control_for(heating, period)



### Cooling

Compared to the PNC, the PINC produces a smoother cooling response, with $q_{HVAC}$ bounded roughly between -500 and +750 W·m^-2. The integral action builds the correction progressively and improves disturbance rejection, which keeps indoor temperature more consistently inside the comfort band. 


In [ ]:
#################
# Cooling       #
#################
cooling = """
Tisp  = 20      # indoor setpoint temperature, °C
Δθ    = 5       # temperature deadband, °C
Kpp   = 1e2     # controller gain
Ki    = 0.05
q_max = Q_MAX

error = Tisp - θ_exp.iloc[k - 1]['c0θ5']
u_pi  = Kpp * error + Ki * integral[0]

if θ_exp.iloc[k - 1]['c0θ5'] < Tisp + Δθ:
    u.iloc[k]['c0θ5'] = 0
    integral[0]       = 0.0
else:
    u_sat = float(np.clip(u_pi, -q_max, 0))
    u.iloc[k]['c0θ5'] = u_sat
    if -q_max < u_pi < 0:
        integral[0] = integral[0] + error * dt
"""
period = ['07-01 12:00:00', '07-03 12:00:00']
control_for(cooling, period)



### Heating and cooling

The PINC reduces the bidirectional swings seen with the PNC under summer mixed heating-cooling conditions. Its integral memory damps oscillatory switching and leads to a more progressive and energy-efficient correction. 


In [ ]:
####################################
# Heating and cooling with deadband #
####################################
heat_cool = """
Tisp  = 20      # indoor setpoint temperature, °C
Δθ    = 5       # temperature deadband, °C
Kpp   = 3e2     # controller gain
Ki    = 0.1
q_max = Q_MAX

error = Tisp - θ_exp.iloc[k - 1]['c0θ5']

if Tisp < θ_exp.iloc[k - 1]['c0θ5'] < Tisp + Δθ:
    u.iloc[k]['c0θ5'] = 0
    integral[0]       = 0.0
else:
    u_pi  = Kpp * error + Ki * integral[0]
    u_sat = float(np.clip(u_pi, -q_max, q_max))
    u.iloc[k]['c0θ5'] = u_sat
    if -q_max < u_pi < q_max:
        integral[0] = integral[0] + error * dt
"""
period = ['05-01 12:00:00', '05-03 12:00:00']
control_for(heat_cool, period)


### Solar protection

During July solar protection, the PINC keeps indoor temperature within a slightly narrower band than the PNC while $q_{HVAC}$ oscillates around -500 W·m^-2. The smoother power profile suggests that the controller handles sustained solar disturbances more gradually. 


In [ ]:
#####################
# Solar protection  #
#####################
solar_protection = """
Tisp  = 20      # indoor setpoint temperature, °C
Δθ    = 1       # temperature deadband, °C
Kpp   = 3e2     # controller gain
q_max = Q_MAX

if θ_exp.iloc[k - 1]['c0θ5'] < Tisp + Δθ:
    u.iloc[k]['c0θ5'] = 0
else:
    u.iloc[k]['c0θ5'] = Kpp * (Tisp - θ_exp.iloc[k - 1]['c0θ5'])
    u.iloc[k]['c0θ5'] = min(u.iloc[k]['c0θ5'], 0)
    u.iloc[k]['c0θ0'] *= 0.1
"""
period = ['05-01 12:00:00', '05-03 12:00:00']
control_for(solar_protection, period)


### Passive cooling

In passive cooling mode, the PINC correctly clamps $q_{HVAC} \approx 0$ when natural ventilation is enough to maintain comfort. The saturation mechanism prevents the integrator from generating unnecessary heating near the comfort boundary. 

In [ ]:


##############################
# Passive cooling            #
##############################
passive_cooling = """
Tisp  = 20.0    # indoor setpoint temperature, °C
Δθ    = 1.0     # temperature deadband, °C
q_max = Q_MAX

ACH_n   = 10.0
V_dot_n = ACH_n / 3600.0 * V_a
G_free  = materials['air']['rho'] * materials['air']['c'] * V_dot_n
q_free  = G_free * (u.iloc[k - 1]['c0q0'] - θ_exp.iloc[k - 1]['c0θ5'])
q_free  = float(np.clip(q_free, -q_max, q_max))

if θ_exp.iloc[k - 1]['c0θ5'] < Tisp + Δθ:
    u.iloc[k]['c0θ5'] = 0
else:
    u.iloc[k]['c0θ5'] = 0
    if u.iloc[k - 1]['c0q0'] < Tisp:
        u.iloc[k]['c0θ5'] = q_free
        u.iloc[k]['c0θ0'] *= 0.1
"""
period = ['05-01 12:00:00', '05-03 12:00:00']
control_for(passive_cooling, period)


# 11 Conclusion

The complete thermal modelling and control analysis of the 30 m² apartment in Grenoble shows that the building is strongly heating-dominated, with an overall thermal conductance of $H_{total} = 63.47$ W/K and ventilation losses $G_{12} = 30$ W/K accounting for 47% of the total. The window is the second-largest loss path and also the primary entry point for solar gains, contributing up to 540 W at peak summer irradiance. 

Under the most unfavorable conditions, the peak winter heating load reaches about 1904 W, while the peak summer cooling load reaches about 1186 W. This confirms that the apartment requires significantly more heating support than cooling support. 

All five eigenvalues of the state matrix are real and negative, which confirms asymptotic stability of the uncontrolled system. The DAE and state-space models agree to an error on the order of 10^-15 °C, which validates the matrix assembly and solver implementation. 

With a perfect controller using $K_p = 10^4$ W/K, indoor temperature remains within ±0.1 °C of the setpoint in both seasons, switching between 20 °C during occupied hours and 16 °C at night. This accuracy comes at the cost of a much smaller stable time step, reduced from 1800 s to 20 s, which highlights the trade-off between control precision and simulation efficiency. 

Free-running simulations show that passive behavior alone is insufficient for comfort, since indoor temperature reaches 33.5 °C in summer and -2.9 °C in winter without HVAC. Among the nonlinear controllers, the PNC maintains comfort but exhibits steady-state offset and chattering, while the PINC removes the offset and produces smoother, more robust behavior through integral action and anti-windup. 

The PINC is therefore the preferred strategy for practical implementation because it combines zero steady-state error, reduced energy waste, and better response during solar gain spikes and passive cooling transitions. A natural extension of this work would be to explore predictive control strategies that use weather forecasts to pre-condition the building thermal mass and reduce peak HVAC loads further. 